# ZaureLink Fine-Tuning: Gemma 4 E2B for Hausa↔English Translation

**Track:** Gemma for Local Languages & Literacy — Build With Gemma: GDG on Campus ABU Zaria  
**Environment:** Google Colab (NVIDIA Tesla T4, 16GB VRAM)  
**Base Model:** `google/gemma-4-E2B-it` (5.15B parameters)  
**Method:** QLoRA via Unsloth + TRL SFTTrainer

This notebook fine-tunes Gemma 4 E2B on a custom 1,048-record Hausa↔English conversational dataset for offline, on-device translation. The trained model is designed to run entirely on low-end Android hardware (≤4GB RAM, Snapdragon 4-series) via LiteRT-LM.

## 1. Environment Setup & Hardware Verification

Installs core dependencies:
- **Unsloth** — memory-efficient QLoRA training framework
- **xformers** — optimized attention kernels
- **TRL** — Hugging Face's supervised fine-tuning trainer
- **PEFT** — parameter-efficient fine-tuning adapters
- **typing-extensions / pydantic** — fixes `ImportError: typing_extensions.Sentinel` (see [replication guide §5.3](../docs/replication_guide.md))

Mounts Google Drive for persistent storage and verifies GPU hardware.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

!pip install --upgrade typing-extensions pydantic pydantic-core

import torch
from google.colab import drive
import os

print("Requesting Google Drive access to persist data and models...")
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ZaureLink'
DATA_DIR = f'{PROJECT_DIR}/dataset'
MODEL_DIR = f'{PROJECT_DIR}/models'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"✅ Project directories initialized at: {PROJECT_DIR}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU detected: {gpu_name}")
    print(f"✅ Total VRAM: {gpu_mem_gb:.2f} GB")

    if "T4" not in gpu_name:
        print("⚠️  Note: You are not on a T4 GPU. Operations will work, but memory behavior may differ from expectations.")
    elif gpu_mem_gb < 14.0:
        print("⚠️  Warning: Your T4 GPU is reporting unusually low available VRAM. Expect potential Out-Of-Memory errors.")
else:
    raise SystemError("❌ No GPU detected! Stop here. Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then restart the session.")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-e8ouu49y/unsloth_8c987a1469064f158487ccdfeba46aac
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-e8ouu49y/unsloth_8c987a1469064f158487ccdfeba46aac
  Resolved https://github.com/unslothai/unsloth.git to commit 47fa4ca6c156c8b9c05663354ee52704213a3cce
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 21.8 MB/s eta 0:00:00
  

## 2. Data Loading & Chat Template Formatting

Loads the final training dataset (`zaurelink_training_data.jsonl` — 1,048 records) from Google Drive. Each record is a multi-turn conversation with system prompt, user utterances (tagged with `spoken_language`), and expected model translations.

The records are formatted into Gemma's native chat template using `tokenizer.apply_chat_template()`, then split 90/10 into training (943) and validation (105) sets with seed 42.

> **Dataset composition:** 297 Market×EN + 297 Market×HA + 227 Campus×EN + 227 Campus×HA records, covering both translation and pass-through scenarios across all 4 system prompt configurations.

In [ ]:
import unsloth
import os
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(hf_token)
    print("✅ Successfully logged into Hugging Face.")
except userdata.SecretNotFoundError:
    raise ValueError("❌ 'HF_TOKEN' not found.")

model_name = "google/gemma-4-E2B-it"
print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

dataset_path = "/content/drive/MyDrive/ZaureLink/dataset/zaurelink_training_data.jsonl"
print(f"Loading dataset from {dataset_path}...")
raw_dataset = load_dataset("json", data_files=dataset_path, split="train")

def format_chat(example):
    formatted_text = tokenizer.apply_chat_template(
        example['conversations'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted_text}

print("Applying Gemma chat template to all records...")
formatted_dataset = raw_dataset.map(format_chat)

print("Splitting dataset into 90% training and 10% validation...")
split_dataset = formatted_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"✅ Data Preparation Complete! Training: {len(train_dataset)} | Validation: {len(val_dataset)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Successfully logged into Hugging Face.
Loading tokenizer for google/gemma-4-E2B-it...
Loading dataset from /content/drive/MyDrive/ZaureLink/dataset/zaurelink_training_data.jsonl...
Applying Gemma chat template to all records...


Map:   0%|          | 0/1048 [00:00<?, ? examples/s]

Splitting dataset into 90% training and 10% validation...
✅ Data Preparation Complete! Training: 943 | Validation: 105


## 3. Base Model & QLoRA Configuration

Loads `google/gemma-4-E2B-it` in 4-bit quantization via Unsloth and injects LoRA adapters.

| Parameter | Value |
|---|---|
| Quantization | 4-bit NormalFloat (NF4) |
| Max Sequence Length | 2048 tokens |
| LoRA Rank ($r$) | 16 |
| LoRA Alpha ($\alpha$) | 16 |
| LoRA Dropout | 0.05 |
| Bias | None |
| Target Modules | `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj` |
| Trainable Parameters | ~25.3M / 5.15B total (0.49%) |
| Gradient Checkpointing | Unsloth optimized |

In [ ]:
from unsloth import FastLanguageModel
import torch
from google.colab import userdata

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-4-E2B-it",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=userdata.get('HF_TOKEN'),
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

==((====))==  Unsloth 2026.7.5: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


trainable params: 25,337,856 || all params: 5,148,515,872 || trainable%: 0.4921


## 4. Supervised Fine-Tuning

Runs QLoRA fine-tuning using TRL's `SFTTrainer` with the following configuration:

| Parameter | Value |
|---|---|
| Epochs | 1 |
| Per-device Batch Size | 2 |
| Gradient Accumulation Steps | 4 |
| **Effective Batch Size** | **8** |
| Total Steps | 118 |
| Learning Rate | 2e-4 (linear decay) |
| Warmup Steps | 10 |
| Weight Decay | 0.05 |
| Optimizer | 8-bit AdamW |
| Precision | float32 (auto-selected by Unsloth for Gemma4 on T4) |

Training converges from loss 1.07 → 0.02 in ~14 minutes.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import gc

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.05,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="/tmp/zaurelink_outputs",
        report_to="none",
    ),
)

try:
    trainer_stats = trainer.train()
    print("\n✅ Training Complete!")
except Exception as e:
    if "PicklingError" in str(e) or "SFTConfig" in str(e):
        print("\n✅ Training Complete! (Ignored known Hugging Face save bug)")
    else:
        raise e

del trainer
gc.collect()
torch.cuda.empty_cache()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/943 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 943 | Num Epochs = 1 | Total steps = 118
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 25,337,856 of 5,148,515,872 (0.49% trained)


Step,Training Loss
10,1.068472
20,0.539815
30,0.270104
40,0.081423
50,0.030739
60,0.023433
70,0.026289
80,0.026165
90,0.024306
100,0.026192


Unsloth: Restored added_tokens_decoder metadata in /tmp/zaurelink_outputs/checkpoint-118/tokenizer_config.json.



✅ Training Complete! (Ignored known Hugging Face save bug)


## 5. Evaluation & Inference Testing

Switches the model to inference mode and runs two evaluation suites:

### Part 1 — Validation Set Testing
Tests against held-out validation samples, comparing expected vs. generated translations.

### Part 2 — Out-of-Distribution & Multi-Turn Testing
Runs 8 scenarios the model has never seen, testing generalization across:
- **Market Mode:** Haggling, units/measurements, blessings, multi-turn price negotiation
- **Campus Mode:** Clinical symptoms, dosage instructions, transport negotiation, multi-turn clinical context

System prompts for both Market and Campus modes are defined here, matching the canonical prompts from `canonical_prompts.py`.

In [ ]:
import random
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

SYS_PROMPT_MARKET_EN = """You are a translation engine operating in Market Mode. A trader and a customer are having a live conversation. The customer requires output in English; the trader requires output in Hausa. Detect which language the current speaker actually used -- either party may speak Hausa or English in any turn, and may mix both within a single utterance. Apply these rules in order: 1. If the speaker's language already matches what the listener on the other end requires, relay their words accurately without altering the language -- do not translate language that doesn't need translating. 2. If it doesn't match, translate it into the listener's required language. 3. If the utterance mixes Hausa and English within a single sentence, treat the full utterance as one unit and translate the entire meaning into the listener's required language -- do not output a half-translated sentence. When translating, resolve the following into their intended meaning rather than translating word-for-word: local currency shorthand and numeric conventions (e.g. "dari biyar" means 500, "dubu biyu" means 2,000 -- output the actual number with "naira"); negotiation idiom and bargaining language (e.g. "farashi na karshe" means "final price," not literally "my last price"); market-specific units of measurement (e.g. "mudu," "tiya," "roba" -- translate to the nearest standard equivalent or keep the local term with a contextual gloss if no standard equivalent exists); culturally embedded greetings, blessings, and social formulas (e.g. "Allah ya kara albarka" -> "May God increase your blessings" when directed at the listener; relay unchanged when it is a formulaic greeting that both parties understand). Use the conversation so far to resolve references -- prices, items, or quantities mentioned in earlier turns -- and to keep each speaker's tone and register (formal, casual, urgent) consistent across turns. If a speaker's intent is genuinely ambiguous or unclear, translate your best interpretation of what was said rather than guessing at unstated meaning. Output only the translated or relayed result, nothing else -- no commentary, no explanations, no metadata."""

SYS_PROMPT_CAMPUS_HA = """You are a translation engine operating in Campus Mode, covering all everyday interactions outside the market. A student (the app user) is having a live conversation with another party -- a driver, a chemist, a fellow student, a lecturer, administrative staff, or anyone else. The student requires output in Hausa; the other party requires output in English. Detect which language the current speaker actually used -- either party may speak Hausa or English in any turn, and may mix both within a single utterance. Apply these rules in order: 1. If the speaker's language already matches what the listener on the other end requires, relay their words accurately without altering the language -- do not translate language that doesn't need translating. 2. If it doesn't match, translate it into the listener's required language. 3. If the utterance mixes Hausa and English within a single sentence, treat the full utterance as one unit and translate the entire meaning into the listener's required language -- do not output a half-translated sentence. When translating, apply domain-appropriate resolution rather than word-for-word phrasing: transport fare amounts, route names, and vehicle references (e.g. "Keke" means tricycle/auto-rickshaw, "Okada" means motorcycle taxi -- use the local term with its meaning if no precise English equivalent exists; resolve "dari biyu" in a fare context to "two hundred naira"); clinical symptom descriptions, body-part references, medication names, and duration/severity expressions accurately (e.g. "jikina yana zafi" -> "my body aches," not literally "my body is hot/painful"; preserve dosage instructions exactly -- do not paraphrase quantities, frequencies, or medication names); academic and administrative terminology (resolve course registration terms, department names, exam-related language, NYSC documentation terminology, and fee payment instructions into their standard English or Hausa equivalents as appropriate); general social formulas (handle greetings, blessings, and culturally embedded social formulas appropriately -- e.g. elaborate Hausa greeting protocols should be translated with their social intent preserved, not reduced to a single "hello"; relay formulaic responses like "Alhamdulillahi" unchanged when both parties understand them). Use the conversation so far to resolve references -- a fare, a symptom, a course name, or a document mentioned in earlier turns -- and to keep each speaker's tone and register (formal with a lecturer, casual with a peer, clinical with a chemist) consistent across turns. If a speaker's intent is genuinely ambiguous or unclear, translate your best interpretation of what was said rather than guessing at unstated meaning. Output only the translated or relayed result, nothing else -- no commentary, no explanations, no metadata."""

def generate_response(messages):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text=prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
    response = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    return response

print("="*80)
print("🔍 PART 1: VALIDATION SET TRIPLES (Checking against held-out 10% data)")
print("="*80)

random.seed(42)
sample_indices = random.sample(range(len(val_dataset)), 2)

for i, idx in enumerate(sample_indices):
    record = val_dataset[idx]
    convs = record['conversations']

    input_conv = convs[:-1]
    expected_output = convs[-1]['content']

    actual_output = generate_response(input_conv)
    last_input = input_conv[-1]['content']

    print(f"\n--- Validation Sample {i+1} ---")
    print(f"🔹 INPUT (Last Turn) : {last_input}")
    print(f"✅ EXPECTED          : {expected_output}")
    print(f"🤖 ACTUAL (Generated): {actual_output}")

mkt_test_1 = [
    {"role": "system", "content": SYS_PROMPT_MARKET_EN},
    {"role": "user", "content": "Gaskiya ba zan sayar a dari uku ba, sai dari hudu."}
]
mkt_test_2 = [
    {"role": "system", "content": SYS_PROMPT_MARKET_EN},
    {"role": "user", "content": "I want to buy three mudu of rice. How much is it?"}
]
mkt_test_3 = [
    {"role": "system", "content": SYS_PROMPT_MARKET_EN},
    {"role": "user", "content": "Allah ya bada sa'a, amma kudin yayi yawa."}
]
mkt_multi_turn = [
    {"role": "system", "content": SYS_PROMPT_MARKET_EN},
    {"role": "user", "content": "Nawa ne wannan takalmin?"},
    {"role": "model", "content": "How much are these shoes?"},
    {"role": "user", "content": "Takalmin dubu biyar ne, na asali ne."},
    {"role": "model", "content": "The shoes are five thousand naira, they are original."},
    {"role": "user", "content": "Haba, can you reduce it to three thousand?"}
]

cmp_test_1 = [
    {"role": "system", "content": SYS_PROMPT_CAMPUS_HA},
    {"role": "user", "content": "Ina jin ciwon kai mai tsanani kuma jikina yana zafi."}
]
cmp_test_2 = [
    {"role": "system", "content": SYS_PROMPT_CAMPUS_HA},
    {"role": "user", "content": "You need to take two of these tablets every eight hours after eating."}
]
cmp_test_3 = [
    {"role": "system", "content": SYS_PROMPT_CAMPUS_HA},
    {"role": "user", "content": "Dan Allah ka kaini gate, nawa ne kudin Keke?"}
]
cmp_multi_turn = [
    {"role": "system", "content": SYS_PROMPT_CAMPUS_HA},
    {"role": "user", "content": "I have been vomiting since last night."},
    {"role": "model", "content": "Ina ta amai tun daren jiya."},
    {"role": "user", "content": "Sannu, ka ci wani abu ne da ya baci?"},
    {"role": "model", "content": "Sorry, did you eat something bad?"},
    {"role": "user", "content": "No, just the rice from the cafeteria. Give me something to stop it."}
]

print("\n" + "="*80)
print("🧪 PART 2: OUT-OF-DISTRIBUTION & MULTI-TURN TESTING")
print("="*80)

def test_model(messages, test_name):
    print(f"\n{'-'*60}\nTEST: {test_name}\n{'-'*60}")
    for msg in messages[1:]:
        print(f"[{msg['role'].upper()}]: {msg['content']}")
    response = generate_response(messages)
    print(f"\n✨ [GENERATED]: {response}")

test_model(mkt_test_1, "Market Single 1 - Hausa to English (Haggling)")
test_model(mkt_test_2, "Market Single 2 - English to Hausa (Units)")
test_model(mkt_test_3, "Market Single 3 - Hausa to English (Blessing)")
test_model(mkt_multi_turn, "Market MULTI-TURN - Context Resolution")

test_model(cmp_test_1, "Campus Single 1 - Hausa to English (Symptoms)")
test_model(cmp_test_2, "Campus Single 2 - English to Hausa (Dosage)")
test_model(cmp_test_3, "Campus Single 3 - Hausa to English (Transport)")
test_model(cmp_multi_turn, "Campus MULTI-TURN - Clinical Context Resolution")

🔍 PART 1: VALIDATION SET TRIPLES (Checking against held-out 10% data)

--- Validation Sample 1 ---
🔹 INPUT (Last Turn) : Suna tafiya lafiya lau.
✅ EXPECTED          : Suna tafiya lafiya lau.
🤖 ACTUAL (Generated): They are traveling well.

--- Validation Sample 2 ---
🔹 INPUT (Last Turn) : Zan je in mika masa assignment dina.
✅ EXPECTED          : I will go and hand in my assignment to him.
🤖 ACTUAL (Generated): I will go and give him my assignment.

🧪 PART 2: OUT-OF-DISTRIBUTION & MULTI-TURN TESTING

------------------------------------------------------------
TEST: Market Single 1 - Hausa to English (Haggling)
------------------------------------------------------------
[USER]: Gaskiya ba zan sayar a dari uku ba, sai dari hudu.

✨ [GENERATED]: Honestly, I won't sell it for three, it's four.

------------------------------------------------------------
TEST: Market Single 2 - English to Hausa (Units)
------------------------------------------------------------
[USER]: I want to buy thre

## 6. LoRA Merge & LiteRT Export (Attempt 1)

Merges the trained LoRA adapters into the base model weights at 16-bit precision, then attempts to export to LiteRT-LM format (`.litertlm`) for Android deployment.

> **⚠️ This cell fails** due to a Protobuf version mismatch (`gencode 6.31.1` vs `runtime 5.29.6`). See Section 7 for the retry, and [notebook 03](./03_litertlm_export.ipynb) for the successful Kaggle TPU export.

In [ ]:
import torch
import gc
import subprocess
import os

print("1. Installing LiteRT-Torch toolchain...")
subprocess.run(["pip", "install", "litert-torch-nightly", "--break-system-packages"], check=True)

print("\n2. Merging LoRA adapter into base weights...")
merged_dir = "/tmp/zaurelink_merged"
model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")

del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

print("\n3. Compiling PyTorch checkpoint to Android .litertlm format...")
output_dir = "/content/drive/MyDrive/ZaureLink/models"

export_cmd = [
    "litert-torch", "export_hf",
    f"--model={merged_dir}",
    f"--output_dir={output_dir}",
    "--quantization_recipe=dynamic_wi4_afp32",
    "--cache_length=1536",
    "--prefill_lengths=128,256",
    "--externalize_embedder",
    "--bundle_litert_lm",
    "--use_jinja_template"
]

result = subprocess.run(export_cmd, capture_output=True, text=True)

if result.returncode != 0:
    print("❌ Export failed! Error log:")
    print(result.stderr)
else:
    print(f"✅ Export successful! Your Android model is ready at: {output_dir}")
    for file in os.listdir(output_dir):
        print(f"   - {file}")

1. Installing LiteRT-Torch toolchain...

2. Merging LoRA adapter into base weights...


config.json:   0%|          | 0.00/5.04k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /tmp/zaurelink_merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Splitting model.safetensors (size: 9.54 GB)...


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [20:14<00:00, 1214.86s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 5/5 [02:18<00:00, 27.71s/it]


Unsloth: Regenerating safetensors index...
Unsloth: Merge process complete. Saved to `/tmp/zaurelink_merged`

3. Compiling PyTorch checkpoint to Android .litertlm format...
❌ Export failed! Error log:
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Traceback (most recent call last):
  File "/usr/local/bin/litert-torch", line 5, in <module>
    from litert_torch.cli import main
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/cli.py", line 20, in <module>
    from litert_torch.generative.export_hf import export as hf_export_lib
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/generative/export_hf/export.py",

## 7. Export Retry (Attempt 2)

Upgrades Protobuf and retries the LiteRT export.

> **⚠️ This cell also fails** — `torchao` encounters a shared library loading error (`_C_mxfp8.cpython-310`). The Colab T4 environment lacks sufficient system RAM to hold the merged FP32 checkpoint and quantization buffers concurrently.

In [ ]:
import subprocess
import os

print("1. Upgrading Protobuf to match LiteRT requirements...")
subprocess.run(["pip", "install", "--upgrade", "protobuf", "--break-system-packages"], check=True)

print("\n2. Retrying compilation to Android .litertlm format...")
merged_dir = "/tmp/zaurelink_merged"
output_dir = "/content/drive/MyDrive/ZaureLink/models"

os.makedirs(output_dir, exist_ok=True)

export_cmd = [
    "litert-torch", "export_hf",
    f"--model={merged_dir}",
    f"--output_dir={output_dir}",
    "--quantization_recipe=dynamic_wi4_afp32",
    "--cache_length=1536",
    "--prefill_lengths=128,256",
    "--externalize_embedder",
    "--bundle_litert_lm",
    "--use_jinja_template"
]

result = subprocess.run(export_cmd, capture_output=True, text=True)

if result.returncode != 0:
    print("❌ Export failed! Error log:")
    print(result.stderr)
else:
    print(f"✅ Export successful! Your Android model is ready at: {output_dir}")
    for file in os.listdir(output_dir):
        print(f"   - {file}")

1. Upgrading Protobuf to match LiteRT requirements...

2. Retrying compilation to Android .litertlm format...
❌ Export failed! Error log:
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so

Loading weights:  37%|███▋      | 754/2011 [00:13<01:48, 11.60it/s] 


## 8. Emergency Model Weight Rescue

Rescues the merged 16-bit model weights (~9.54 GB safetensors) from the volatile `/tmp` directory to persistent Google Drive storage before the Colab runtime recycles.

These rescued weights are later uploaded as a Kaggle Dataset and used by [notebook 03](./03_litertlm_export.ipynb) on a Kaggle TPU v5e-8 instance, where the export succeeds.

In [ ]:
import shutil

shutil.copytree('/tmp/zaurelink_merged', '/content/drive/MyDrive/ZaureLink/models/zaurelink_merged_safe', dirs_exist_ok=True)
print("Model successfully rescued to Google Drive!")

Model successfully rescued to Google Drive!


## 9. Export Attempts 3–4 (Colab — Failed)

Final attempts to run `litert-torch export_hf` on Colab, including with `--experimental_lightweight_conversion`. Both fail due to insufficient system RAM and eventual Google Drive disconnection.

> **Resolution:** The export was successfully completed on a **Kaggle TPU v5e-8** instance (selected for its ~90GB system RAM headroom). See [notebook 03 — LiteRT-LM Export](./03_litertlm_export.ipynb) for the successful pipeline.

---

*Cells 6–10 are preserved as-is to document the real engineering journey: the Colab→Kaggle migration was a critical architectural decision driven by the memory wall encountered here.*

In [ ]:
from google.colab import drive
import os
import shutil
import subprocess

drive.mount('/content/drive')

subprocess.run(["pip", "install", "litert-torch-nightly", "--break-system-packages"], check=True)
subprocess.run(["pip", "install", "--upgrade", "protobuf", "--break-system-packages"], check=True)

subprocess.run(["cp", "-r", "/content/drive/MyDrive/ZaureLink/models/zaurelink_merged_safe", "/content/local_model"], check=True)

os.makedirs("/content/local_output", exist_ok=True)

subprocess.run([
    "litert-torch", "export_hf",
    "--model=/content/local_model",
    "--output_dir=/content/local_output",
    "--quantization_recipe=dynamic_wi4_afp32",
    "--cache_length=1536",
    "--prefill_lengths=128,256",
    "--externalize_embedder",
    "--bundle_litert_lm",
    "--use_jinja_template"
])

for file in os.listdir("/content/local_output"):
    if file.endswith(".litertlm"):
        shutil.copy(f"/content/local_output/{file}", f"/content/drive/MyDrive/ZaureLink/models/{file}")
        print(f"✅ SUCCESS: {file} copied to your Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install litert-torch-nightly --break-system-packages
!pip install --upgrade protobuf --break-system-packages

!rm -rf /content/local_model
!rm -rf /content/local_output
!cp -r /content/drive/MyDrive/ZaureLink/models/zaurelink_merged_safe /content/local_model
!mkdir -p /content/local_output

!litert-torch export_hf \
  --model=/content/local_model \
  --output_dir=/content/local_output \
  --quantization_recipe=dynamic_wi4_afp32 \
  --cache_length=1536 \
  --prefill_lengths=128,256 \
  --externalize_embedder \
  --bundle_litert_lm \
  --use_jinja_template \
  --experimental_lightweight_conversion \

!cp /content/local_output/*.litertlm /content/drive/MyDrive/ZaureLink/models/
!ls -lh /content/drive/MyDrive/ZaureLink/models/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
============== Export Configuration ==============
aot_backend            : None
aot_compilation_config_dict : None
aot_soc_model          : None
assistant_model        : None
auto_model_override    : None
batch_size             : 1
bundle_litert_lm       : True
cache_implementation   : 'LiteRTLMCache'
cache_length           : 1536
cache_length_dim       : None
enable_dynamic_shape   : False
enable_gpu_dynamic_cache : False
enable_gpu_dynamic_prefill : Fals